# 人手評価データで翻訳の品質推定モデルを学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_machine_translation/mt_qe_train.ipynb)

NLP クックブック 27-08 ／ 機械翻訳

参照訳なしで翻訳の品質を見積もりたいが、単純な長さ比や埋め込みのコサイン類似度だけでは細かい優劣が当たらない。人手評価データ (WMT 2022) を用いて、多言語文埋め込みの交差表現 $[u, v, |u - v|, u \odot v]$ と文字長比の特徴から品質推定モデルを学習し、学習なし手法と比べる。

In [1]:
!pip install -q datasets sentence-transformers

In [2]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy.stats import kendalltau, spearmanr

# WMT 2022 の人手評価（前のレシピと同じ英日翻訳データ）
df = load_dataset("RicardoRei/wmt-sqm-human-evaluation", split="train").to_pandas()
df = df[(df["lp"] == "en-ja") & (df["src"].str.len().between(20, 200))].reset_index(drop=True)
print(f"件数: {len(df)} / ユニークな原文数: {df['src'].nunique()} / システム数: {df['system'].nunique()}")

件数: 13897 / ユニークな原文数: 1023 / システム数: 13


In [3]:
from sentence_transformers import SentenceTransformer

# 多言語文埋め込みで原文ベクトル u と訳文ベクトル v を計算する（CPU で約 15〜30 秒）
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
u = embedder.encode(["query: " + s for s in df["src"]], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
v = embedder.encode(["query: " + s for s in df["mt"]], normalize_embeddings=True, batch_size=64, show_progress_bar=False)

# COMET や TransQuest で用いられる交差表現と文字長比特徴を作る
src_len = df["src"].str.len().to_numpy(dtype=float)
mt_len = df["mt"].str.len().to_numpy(dtype=float)
ratio = mt_len / np.maximum(src_len, 1.0)
cos_sim = (u * v).sum(axis=1)

X_cross = np.hstack([u, v, np.abs(u - v), u * v])
X_len = np.column_stack([np.log1p(src_len), np.log1p(mt_len), ratio, np.log(np.maximum(ratio, 1e-6))])
y = df["score"].to_numpy()
groups = df["src"].to_numpy()
print("特徴量の次元:", X_cross.shape[1], "(埋め込み交差表現) +", X_len.shape[1], "(長さ比特徴)")

特徴量の次元: 1536 (埋め込み交差表現) + 4 (長さ比特徴)


In [4]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# 同一原文に対する複数システムの訳が訓練と評価にまたがらないよう GroupKFold で分割する
gkf = GroupKFold(n_splits=5)
alphas = np.logspace(-1, 4, 11)
oof = {k: np.empty(len(df)) for k in ("長さ比ルール", "長さ特徴のみ (RidgeCV)", "交差埋め込みのみ (RidgeCV)", "交差埋め込み + 長さ特徴 (RidgeCV)")}

for tr, te in gkf.split(X_cross, y, groups):
    assert set(groups[tr]).isdisjoint(set(groups[te]))
    med_ratio = np.median(ratio[tr])
    oof["長さ比ルール"][te] = -np.abs(np.log(np.maximum(ratio[te], 1e-6) / med_ratio))

    # 長さ特徴のみ訓練データで標準化し、L2 正規化済み埋め込み特徴はスケールを保つ
    scaler = StandardScaler().fit(X_len[tr])
    Xtr_l, Xte_l = scaler.transform(X_len[tr]), scaler.transform(X_len[te])
    oof["長さ特徴のみ (RidgeCV)"][te] = RidgeCV(alphas=alphas).fit(Xtr_l, y[tr]).predict(Xte_l)
    oof["交差埋め込みのみ (RidgeCV)"][te] = RidgeCV(alphas=alphas).fit(X_cross[tr], y[tr]).predict(X_cross[te])
    oof["交差埋め込み + 長さ特徴 (RidgeCV)"][te] = RidgeCV(alphas=alphas).fit(
        np.hstack([X_cross[tr], Xtr_l]), y[tr]
    ).predict(np.hstack([X_cross[te], Xte_l]))

rows = []
for name, pred in [("埋め込みの類似度 (学習なし)", cos_sim), *oof.items()]:
    sp = spearmanr(y, pred)
    kt = kendalltau(y, pred)
    rows.append({"手法": name, "Spearman": sp.statistic, "Kendall": kt.statistic, "p 値 (Spearman)": sp.pvalue})
res_df = pd.DataFrame(rows)
print(res_df.to_string(index=False, formatters={
    "Spearman": "{:+.3f}".format, "Kendall": "{:+.3f}".format, "p 値 (Spearman)": "{:.2e}".format
}))

# 原文単位の対応のあるブートストラップで、交差埋め込み + 長さ特徴とベースラインの差の 95% 信頼区間を確認する
rng = np.random.default_rng(0)
src_idx = {s: np.flatnonzero(groups == s) for s in np.unique(groups)}
uniq_srcs = list(src_idx)
p_emb = oof["交差埋め込み + 長さ特徴 (RidgeCV)"]
p_len = oof["長さ特徴のみ (RidgeCV)"]
diff_vs_cos, diff_vs_len = [], []
for _ in range(300):
    ix = np.concatenate([src_idx[uniq_srcs[i]] for i in rng.integers(0, len(uniq_srcs), len(uniq_srcs))])
    s_emb = spearmanr(y[ix], p_emb[ix]).statistic
    diff_vs_cos.append(s_emb - spearmanr(y[ix], cos_sim[ix]).statistic)
    diff_vs_len.append(s_emb - spearmanr(y[ix], p_len[ix]).statistic)
print(f"Spearman 差 (学習モデル − 類似度): {np.mean(diff_vs_cos):+.3f} 95% CI [{np.percentile(diff_vs_cos, 2.5):+.3f}, {np.percentile(diff_vs_cos, 97.5):+.3f}]")
print(f"Spearman 差 (学習モデル − 長さ特徴のみ): {np.mean(diff_vs_len):+.3f} 95% CI [{np.percentile(diff_vs_len, 2.5):+.3f}, {np.percentile(diff_vs_len, 97.5):+.3f}]")

                     手法 Spearman Kendall p 値 (Spearman)
        埋め込みの類似度 (学習なし)   -0.011  -0.008       1.86e-01
                 長さ比ルール   -0.023  -0.015       7.85e-03
       長さ特徴のみ (RidgeCV)   +0.195  +0.133      7.30e-119
     交差埋め込みのみ (RidgeCV)   +0.257  +0.175      1.89e-208
交差埋め込み + 長さ特徴 (RidgeCV)   +0.269  +0.183      1.14e-228
Spearman 差 (学習モデル − 類似度): +0.280 95% CI [+0.240, +0.323]
Spearman 差 (学習モデル − 長さ特徴のみ): +0.075 95% CI [+0.048, +0.103]


- 学習なしの「埋め込みの類似度」や「長さ比ルール」は WMT 2022 のような高品質システムの細かい優劣に対してほぼ無相関だが、同じ文埋め込みから作った交差表現 $[u, v, |u - v|, u \odot v]$ と長さ比特徴で `RidgeCV` を学習すると、有意な正の相関（Spearman 約 +0.27、Kendall 約 +0.18）が得られる。
- 同じ原文に対する複数システムの訳文が訓練側と評価側に分かれると、原文の難易度やトピックを暗記するリークが起きて相関が不当に高く出る。品質推定の評価では必ず原文をグループにした `GroupKFold` 等で分割する。
- 長さ特徴だけでも +0.19 前後の相関が出るが（極端に短い訳や長い訳が低品質になりやすいため）、交差埋め込みを加えることで原文単位のブートストラップ 95% 信頼区間でも有意に上回る。さらに表現力を上げた参照訳なしモデルの実装例（`xlm-roberta-base` のエンドツーエンド微調整）は、姉妹編『[自然言語処理の教科書](https://github.com/mamorlis/nlpbook)』第 1 章の [英日機械翻訳の品質推定](https://github.com/mamorlis/nlpbook/blob/main/notebooks/ch1_qe.ipynb) を参照。
- 参照訳ありの評価指標を文埋め込みから学習する仕組みについては、付録「仕組みを理解するための実装」の「文埋め込みから翻訳の評価指標を自分で学習する」も参照。